# DNA Splice Junction Classification

## Dataset Understanding

In the previous notebook, I introduced the biological concepts behind DNA, RNA splicing, and splice junctions.

In this notebook, I will move from the biological background to the actual dataset used for this project.

My goal here is to understand the structure, contents, and quality of the raw data before making any cleaning or modeling decisions.

## 1. Why Dataset Understanding?

### WHY

Before building a machine learning model, I need to understand the data that the model will learn from.

At this stage, I want to answer questions such as:

- How many observations are present?
- What information does each observation contain?
- What are the target classes?
- How are the DNA sequences represented?
- Are all sequences the same length?
- Are there missing values?
- Are there duplicate sequences?
- Are there any unusual characters or data-quality issues?

I will first inspect the raw data without modifying it.

In [3]:
from pathlib import Path
import pandas as pd
DATA_PATH = Path("../data/raw/splice.data")

## 2. Loading the Raw Dataset

### WHY

The UCI dataset is stored in the `splice.data` file.

Before performing any transformation, I want to load the raw file as it is and inspect its structure.

In [4]:
raw_data = pd.read_csv(
    DATA_PATH,
    header=None,
    skipinitialspace=True
)

raw_data.head()

,0,1,2
0,EI,ATRINS-DONOR-521,CCAGCTGCATCACAGGAGGCCAGCGAGCAGGTCTGTTCCAAGGGCC...
1,EI,ATRINS-DONOR-905,AGACCCGCCGGGAGGCGGAGGACCTGCAGGGTGAGCCCCACCGCCC...
2,EI,BABAPOE-DONOR-30,GAGGTGAAGGACGTCCTTCCCCAGGAGCCGGTGAGAAGCGCAGTCG...
3,EI,BABAPOE-DONOR-867,GGGCTGCGTTGCTGGTCACATTCCTGGCAGGTATGGGGCGGGGCTT...
4,EI,BABAPOE-DONOR-2817,GCTCAGCCCCCAGGTCACCCAGGAACTGACGTGAGTGTCCCCATCC...


### I will first look at the shape and the first few rows of the raw dataset.

In [5]:
print("Shape:", raw_data.shape)
raw_data.head(
)

Shape: (3190, 3)


,0,1,2
0,EI,ATRINS-DONOR-521,CCAGCTGCATCACAGGAGGCCAGCGAGCAGGTCTGTTCCAAGGGCC...
1,EI,ATRINS-DONOR-905,AGACCCGCCGGGAGGCGGAGGACCTGCAGGGTGAGCCCCACCGCCC...
2,EI,BABAPOE-DONOR-30,GAGGTGAAGGACGTCCTTCCCCAGGAGCCGGTGAGAAGCGCAGTCG...
3,EI,BABAPOE-DONOR-867,GGGCTGCGTTGCTGGTCACATTCCTGGCAGGTATGGGGCGGGGCTT...
4,EI,BABAPOE-DONOR-2817,GCTCAGCCCCCAGGTCACCCAGGAACTGACGTGAGTGTCCCCATCC...


## 2.1 Understanding the Raw Columns

### FIND

The raw dataset contains **3,190 observations and 3 columns**.

The first few rows show three pieces of information:

1. **Class** — the splice-junction category.
2. **Instance name** — the identifier associated with the sequence.
3. **DNA sequence** — the nucleotide sequence used for classification.

At this stage, the columns are still represented by numeric column names because we loaded the raw file without assigning names.

The first rows show the `EI` class, followed by an instance identifier such as `ATRINS-DONOR-521`, and a DNA sequence.

### DECIDE

I will assign meaningful column names to make the dataset easier to work with:

- `class`
- `instance_name`
- `sequence`

I will keep `instance_name` for identification and traceability, but it will not be used as a predictive feature.

In [6]:
raw_data.columns = ["class", "instance_name", "sequence"]
raw_data.head()

,class,instance_name,sequence
0,EI,ATRINS-DONOR-521,CCAGCTGCATCACAGGAGGCCAGCGAGCAGGTCTGTTCCAAGGGCC...
1,EI,ATRINS-DONOR-905,AGACCCGCCGGGAGGCGGAGGACCTGCAGGGTGAGCCCCACCGCCC...
2,EI,BABAPOE-DONOR-30,GAGGTGAAGGACGTCCTTCCCCAGGAGCCGGTGAGAAGCGCAGTCG...
3,EI,BABAPOE-DONOR-867,GGGCTGCGTTGCTGGTCACATTCCTGGCAGGTATGGGGCGGGGCTT...
4,EI,BABAPOE-DONOR-2817,GCTCAGCCCCCAGGTCACCCAGGAACTGACGTGAGTGTCCCCATCC...


### After assigning meaningful column names, I want to verify that the structure is correct before continuing with the analysis.

In [7]:
print("Shape:", raw_data.shape)
print("\nColumns:")
print(raw_data.columns.tolist())

print("\nData types:")
print(raw_data.dtypes)

Shape: (3190, 3)

Columns:
['class', 'instance_name', 'sequence']

Data types:
class            object
instance_name    object
sequence         object
dtype: object


## 2.2 Basic Dataset Information

### WHY

I want to understand the data types and whether any values are missing before moving deeper into the dataset.

### DO

I will inspect the dataset information and missing values.

In [8]:
raw_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3190 entries, 0 to 3189
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   class          3190 non-null   object
 1   instance_name  3190 non-null   object
 2   sequence       3190 non-null   object
dtypes: object(3)
memory usage: 74.9+ KB


In [9]:
raw_data.isna().sum()

class            0
instance_name    0
sequence         0
dtype: int64

### FIND

The dataset contains **3,190 observations and 3 columns**.

The columns are:

- `class` - the target class for the DNA sequence.
- `instance_name` - the identifier of the sequence.
- `sequence` - the DNA sequence.

All three columns have an `object` data type because they contain categorical or textual information.

There are **no missing values** in any of the three columns.

The dataset uses approximately **74.9 KB** of memory, so it is small enough to work with comfortably during exploration and model development.

### DECIDE

The raw dataset structure is suitable for further analysis.

I will:

- keep `class` as the target variable,
- keep `sequence` as the main input data,
- retain `instance_name` for identification and traceability,
- avoid using `instance_name` as a model feature,
- preserve the raw data without modifying it at this stage.

Since there are no missing values, no missing-value treatment is required for these columns.

## 2.3 Understanding the Target Classes

### WHY

The target variable determines what the machine learning model needs to predict.

I want to identify all classes present in the dataset and understand how the observations are distributed among them.

In [10]:
raw_data["class"].value_counts()

class
N     1655
IE     768
EI     767
Name: count, dtype: int64

In [11]:
raw_data["class"].value_counts(normalize=True).mul(100).round(2) #percentage

class
N     51.88
IE    24.08
EI    24.04
Name: proportion, dtype: float64

### FIND

The dataset contains three target classes:

| Class | Count | Percentage |
|---|---:|---:|
| **N** | 1,655 | 51.88% |
| **IE** | 768 | 24.08% |
| **EI** | 767 | 24.04% |

The `N` class contains slightly more than half of the observations, while `EI` and `IE` each contain about one quarter of the dataset.

The `EI` and `IE` classes are almost perfectly balanced with only one observation difference between them.

Overall, the target distribution is **moderately imbalanced**, mainly because the `N` class is larger than the two splice-junction classes.

### DECIDE

I will treat this as a **three-class classification problem**.

Because the `N` class is substantially larger than `EI` and `IE`, I will not rely on accuracy alone when evaluating the models.

Later, I will also consider class-wise **precision, recall, F1-score, and the confusion matrix** to understand how well the models perform for each class.

## 2.4 Understanding DNA Sequence Length

### WHY

The `sequence` column contains the DNA sequences that will eventually be converted into machine-learning features.

Before doing that, I want to understand their length.

Sequence length is important because it determines how many k-mers can be generated from each sequence.

In [12]:
sequence_lengths = raw_data["sequence"].str.len()

sequence_lengths.describe()

count    3190.0
mean       60.0
std         0.0
min        60.0
25%        60.0
50%        60.0
75%        60.0
max        60.0
Name: sequence, dtype: float64

In [13]:
sequence_lengths.value_counts().sort_index()

sequence
60    3190
Name: count, dtype: int64

### FIND

All 3,190 sequences have a length of exactly 60 nucleotides.

The summary statistics also show no variation in sequence length:
- Mean: 60
- Standard deviation: 0
- Minimum: 60
- Maximum: 60

This means every observation has the same sequence length.

### DECIDE

Since all sequences have the same length, no sequence-length cleaning or trimming is required.

This is also useful for k-mer feature engineering because every sequence will produce the same number of k-mer positions for a given value of `k`.

## 2.5 Understanding the DNA Characters

### WHY

Before treating the sequences as DNA data, I want to check which nucleotide symbols are present.

A standard DNA sequence contains four bases: `A`, `T`, `G`, and `C`.

Biological datasets can also contain ambiguity symbols when the exact nucleotide is not known. I need to identify these symbols before feature engineering so that I understand the actual structure of the sequence data.

In [14]:
unique_characters = sorted(set("".join(raw_data["sequence"])))

unique_characters

['A', 'C', 'D', 'G', 'N', 'R', 'S', 'T']

### FIND

The sequences contain the following characters:

`A`, `C`, `G`, `T`, `D`, `N`, `R`, and `S`.

The standard DNA nucleotides are:
- `A` — Adenine
- `C` — Cytosine
- `G` — Guanine
- `T` — Thymine

The additional symbols are biological ambiguity codes:
- `D` — A, G, or T
- `N` — A, C, G, or T
- `R` — A or G
- `S` — C or G

Therefore, the dataset contains both standard nucleotide symbols and ambiguity symbols.

### DECIDE

I will preserve these ambiguity symbols rather than replacing them at this stage.

They are part of the original biological sequence representation, so changing them without a specific reason could alter the information contained in the dataset.

I will keep the original sequences intact during dataset understanding and consider their treatment during the later k-mer feature-engineering stage.

## 2.6 Understanding Duplicate Observations

### WHY

I want to check whether the dataset contains duplicate observations.

Duplicate sequences are not automatically errors. In biological datasets, the same sequence can legitimately occur more than once.

However, duplicate sequences are important for this project because the same sequence appearing in both the training and test sets could affect how we evaluate model performance.

I will first identify duplicates without removing anything.

In [15]:
duplicate_rows = raw_data.duplicated().sum()
print("Duplicate complete rows:", duplicate_rows)

Duplicate complete rows: 12


In [16]:
duplicate_sequences = raw_data["sequence"].duplicated().sum()
print("Duplicate sequences:", duplicate_sequences)

Duplicate sequences: 185


### FIND

The dataset contains 12 duplicate complete rows.

It also contains 185 repeated DNA sequences.

The number of repeated sequences is much larger than the number of completely duplicated rows, which means that some sequences appear under different observations.

At this point, I cannot assume that these repeated sequences are errors. I need to check whether the repeated sequences have consistent class labels.

### WHY

A repeated DNA sequence with the same class label is different from a repeated DNA sequence associated with different class labels.

If the same sequence appears with different labels, it creates an ambiguity in the target data and can have a direct effect on model evaluation.

Therefore, I will check the class labels associated with repeated sequences before deciding how to handle them.

In [17]:
sequence_class_counts = (
    raw_data.groupby("sequence")["class"]
    .nunique()
)
conflicting_sequences = sequence_class_counts[sequence_class_counts > 1]
print("Sequences appearing with multiple classes:", len(conflicting_sequences))

Sequences appearing with multiple classes: 1


### FIND

Only 1 unique DNA sequence appears with multiple class labels.

This means the dataset contains a potential label conflict: the same 60-nucleotide sequence has been assigned to more than one target class.

I will inspect the corresponding observations before deciding how this should be handled.

In [18]:
conflicting_sequence = conflicting_sequences.index[0]
raw_data[raw_data["sequence"] == conflicting_sequence]

,class,instance_name,sequence
1021,IE,HUMCYPIIE-ACCEPTOR-12121,CTGAAATTTGTCCCATTCATATCTTGGCAGAGAAGCTCCATGAAGA...
1968,N,HUMCYPIIE-NEG-12121,CTGAAATTTGTCCCATTCATATCTTGGCAGAGAAGCTCCATGAAGA...


### FIND

The investigation shows that the same 60-nucleotide sequence appears twice with different class labels:

- `IE` — `HUMCYPIIE-ACCEPTOR-12121`
- `N` — `HUMCYPIIE-NEG-12121`

The DNA sequence itself is identical in both observations, but the target class is different.

This is a genuine label conflict in the dataset rather than a simple duplicate row.

### DECIDE

I will not automatically remove or relabel these observations.

The conflicting sequence is part of the original UCI dataset, so changing the labels would mean making an unsupported biological assumption.

I will keep the observations in the raw dataset and explicitly account for this conflict when designing the train-test evaluation.

This issue will be important later because placing the same sequence in both training and test sets could lead to misleading evaluation results.

## 2.7 Understanding Instance Names

### WHY

The `instance_name` column identifies individual observations in the original dataset.

I want to check whether any instance names are repeated.

This is mainly a data-integrity check because `instance_name` will not be used as a predictive feature.

In [19]:
duplicate_instance_names = raw_data["instance_name"].duplicated().sum()

print("Duplicate instance names:", duplicate_instance_names)

Duplicate instance names: 12


### FIND

There are 12 duplicate instance names.

This matches the 12 duplicate complete rows found earlier, indicating that these repeated instance names correspond to observations that are duplicated in their entirety.

The `instance_name` column therefore contains duplicate identifiers for these repeated observations.

### DECIDE

I will retain the `instance_name` column for traceability and dataset investigation, but I will not use it as a model feature.

I will also keep the duplicate observations in the raw dataset for now rather than removing them automatically.

Any decision about duplicate handling will be made after considering its effect on model evaluation and the conflicting sequence identified earlier.

## 2.8 Validating the Target Classes

### WHY

I have already examined the distribution of the target classes.

Now I want to verify that every observation belongs to one of the three expected classes: `EI`, `IE`, or `N`.

This is a basic data-integrity check before moving toward data cleaning.

In [20]:
expected_classes = {"EI", "IE", "N"}
actual_classes = set(raw_data["class"].unique())
print("Classes found:", actual_classes)
print("Unexpected classes:", actual_classes - expected_classes)

Classes found: {'IE', 'EI', 'N'}
Unexpected classes: set()


### FIND

The dataset contains exactly three classes:

- `EI`
- `IE`
- `N`

No unexpected class labels were found.

Therefore, the target column is consistent with the expected three-class classification problem.

### DECIDE

I will keep the target labels unchanged.

No class-label correction is required at the dataset-understanding stage.

The project will continue as a multiclass classification problem with `EI`, `IE`, and `N` as the target classes.

## 2.9 Validating the DNA Sequences

### WHY

I have identified the unique characters present in the DNA sequences.

Now I want to verify that every sequence contains only these expected DNA and ambiguity symbols.

This will help identify malformed sequences before moving to the data-cleaning stage.

In [21]:
expected_characters = set(["A", "C", "D", "G", "N", "R", "S", "T"])
unexpected_characters = set()
for sequence in raw_data["sequence"]:
    unexpected_characters.update(set(sequence) - expected_characters)
print("Unexpected characters:", unexpected_characters)

Unexpected characters: set()


In [22]:
has_whitespace = raw_data["sequence"].str.strip().ne(raw_data["sequence"]).sum()
has_lowercase = raw_data["sequence"].str.contains(r"[a-z]", regex=True).sum()
print("Sequences with leading/trailing whitespace:", has_whitespace)
print("Sequences containing lowercase characters:", has_lowercase)

Sequences with leading/trailing whitespace: 0
Sequences containing lowercase characters: 0


### FIND

No sequences contain leading or trailing whitespace.

The sequence data also contains no lowercase characters, so the sequences are consistently formatted.
    
### DECIDE

No formatting correction is required for the DNA sequences.

I will preserve the sequence representation as provided by the dataset.

## 2.10 Dataset Understanding — Summary

### FIND

The dataset contains 3,190 observations with three columns:

- `class` — the target variable
- `instance_name` — the original observation identifier
- `sequence` — the 60-nucleotide DNA sequence

The target contains three classes:

- `N`: 1,655 observations (51.88%)
- `IE`: 768 observations (24.08%)
- `EI`: 767 observations (24.04%)

All sequences have exactly 60 nucleotides.

The sequences contain the standard DNA bases along with the ambiguity symbols `D`, `N`, `R`, and `S`.

There are no missing values, unexpected classes, unexpected sequence characters, or sequence-formatting issues.

The dataset contains 12 duplicate complete rows and 185 repeated sequences. One sequence appears with two different class labels, which represents a label conflict that needs to be considered during evaluation.

### DECIDE

The raw dataset is structurally consistent and does not require immediate cleaning for missing values, sequence length, or invalid characters.

I will retain the original observations and instance names for traceability.

I will not automatically remove duplicate sequences or resolve the conflicting labels because doing so would require an unsupported assumption.

The dataset is now sufficiently understood to move to the data-cleaning stage.